# 02 — Model Integration: Invoke, Stream, Batch

Every LangChain chat model exposes three execution patterns. Pick by **latency profile**, not personal taste.

| Pattern | When | Why |
|---|---|---|
| `invoke` | Classification, formatting, structured output | One result, no UX win from streaming. |
| `stream` | Chat UIs, long-form text | Tokens arrive as generated — perceived latency drops. |
| `batch` | Bulk eval / labelling | Concurrency amortises network round-trips. |

**Providers used in this notebook (mixed deliberately):**

| Cell | Provider | Model | Why |
|---|---|---|---|
| Invoke #1 | `google_genai` | `gemini-2.5-flash` | Default tutorial choice. |
| Invoke #2 | direct class import | `gemini-2.5-flash-lite` | Demonstrates the `ChatGoogleGenerativeAI(...)` path when you need provider-specific kwargs. |
| Invoke #3 → Batch | `groq` | `qwen/qwen3-32b` | Fast, free tier, surfaces `reasoning_content` — useful contrast. |

> Read [`02_model_integration.md`](./02_model_integration.md) alongside this notebook for theory.


## Setup

Load env vars before constructing any model — both `GOOGLE_API_KEY` and `GROQ_API_KEY` must be in your `.env`.


In [17]:
import os
from dotenv import load_dotenv

load_dotenv()


True

## Pattern 1 — `invoke` (single request/response)

Blocks until the full response arrives. Returns one `AIMessage`. Use for everything that doesn't need streaming.

Note the **explicit provider prefix** `google_genai:` — without it, LangChain defaults to Vertex AI and prints a deprecation warning.


In [18]:
from langchain.chat_models import init_chat_model

gemini = init_chat_model("google_genai:gemini-2.5-flash")

response = gemini.invoke("Hello, how are you?")
response.content


"Hello! I'm doing well, thank you for asking.\n\nHow are you today?"

### Direct class import — when you need provider-specific kwargs

The factory `init_chat_model` is great for most cases, but if you want fine-grained control (temperature, generation config, safety thresholds, etc.), import the provider class directly. Same `.invoke()` interface — only construction differs.


In [19]:
from langchain_google_genai import ChatGoogleGenerativeAI

gemini_lite = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

response = gemini_lite.invoke("Hello, how are you?")
response.content


"Hello! I'm doing well, thank you for asking. As a large language model, I don't experience feelings in the same way humans do, but I'm functioning optimally and ready to assist you.\n\nHow about you? How are you doing today?"

### Same interface, different provider

Switch to Groq with one line. The rest of your code is untouched — that's the point of the unified interface.


In [20]:
qwen = init_chat_model("groq:qwen/qwen3-32b")

response = qwen.invoke("Hello, how are you?")
response.content


'<think>\nOkay, the user is asking "Hello, how are you?" which is a common greeting. I need to respond politely and set a friendly tone. I should acknowledge their greeting and offer assistance. Let me make sure the response is welcoming and open-ended to encourage them to ask questions or share what they need help with. Also, keeping it concise but warm.\n</think>\n\nHi there! 😊 I\'m doing great, thanks for asking! How can I assist you today? Whether you have questions, need help with something, or just want to chat, I\'m here for you!'

## Pattern 2 — `stream` (chunk-by-chunk)

Yields `AIMessageChunk` objects as tokens are produced. Concatenate `.content` to build the full text. Used everywhere you want the UI to feel responsive.

Below: a neutral technical prompt so you can watch streaming clearly without long ramps.


In [21]:
for chunk in qwen.stream("Explain how token streaming works in LLMs in 3 short paragraphs."):
    print(chunk.content, end="", flush=True)


<think>
Okay, so I need to explain how token streaming works in LLMs in three short paragraphs. Let me start by recalling what I know about tokens and LLMs.

First, I remember that large language models process text in tokens, which are like the smallest units they can work with. These could be words, parts of words, or even characters, depending on the model's tokenizer. When you input text, it's broken down into these tokens, and the model processes them one at a time. But how does streaming come into play here?

Token streaming probably refers to the model generating text incrementally, outputting tokens as they're predicted rather than waiting to generate the entire response at once. This would be useful for real-time interactions, like chatbots, where you want to see the response as it's being built. But how exactly does this work technically?

I think it involves the model predicting the next token based on the history of previously generated tokens. So each time a new token is g

## Pattern 3 — `batch` (parallel execution)

Sends multiple prompts concurrently through a thread pool. `max_concurrency` caps simultaneous calls — keep it below your provider's per-minute rate limit.

Returns a list of `AIMessage`, one per input prompt, in input order.


In [22]:
responses = qwen.batch(
    [
        "Why do parrots have colorful feathers?",
        "How do airplanes fly?",
        "What is quantum computing?",
    ],
    config={"max_concurrency": 3},
)

for i, r in enumerate(responses):
    print(f"--- prompt {i} ---")
    print(r.content[:300], '...' if len(r.content) > 300 else '')


--- prompt 0 ---
<think>
Okay, so I need to figure out why parrots have such colorful feathers. Let me start by recalling what I know about parrots. They're known for their vibrant colors—greens, blues, reds, yellows, and more. I remember from biology class that animal traits often evolve due to natural selection or ...
--- prompt 1 ---
<think>
Okay, the user is asking, "How do airplanes fly?" Let me break this down. First, I need to explain the basic principles of flight. Airplanes fly due to a combination of forces: lift, weight, thrust, and drag. 

Starting with lift. I remember that lift is generated by the wings. The shape of  ...
--- prompt 2 ---
<think>
Okay, I need to explain what quantum computing is. Let me start by recalling what I know. Quantum computing uses quantum bits or qubits, right? Unlike classical bits which are either 0 or 1, qubits can be in a superposition of both. That means they can represent multiple states at once. But  ...


## Recap & next steps

- `invoke` → one full response, blocking.
- `stream` → token-level iterator.
- `batch` → list of responses, concurrent under the hood.
- `init_chat_model("provider:model")` gives you the same three methods on every provider.
- Direct class imports (`ChatGoogleGenerativeAI(...)`) are an escape hatch for provider-specific options.

**Next**: [`03_tools.ipynb`](./03_tools.ipynb) — the raw mechanics of tool calling that `create_agent` (notebook 01) handled for you.
